# CommuNiche analysis the human breast cancer micro-environment
To facilitate accessibility and reproducibility of our results, we have deposited all datasets used in this study on [Zenodo](https://zenodo.org). Below we will demonstrate the applying of CommuNiche for the exploration of the human breast cancer data.

**Workflow**
1. Load spatial transcriptomics data and LR human reference
2. Construction of communication tensor
3. Tensor decomposition
4. Niche identification
5. Niche interpretation


## Load spatial transcriptomics data and LR human reference
First, let’s load the package and the data. We focus on human breast cancer data from Xenium ST technology (Replication 1), The processed data are available at the [Zenodo data repository] (https://doi.org/10.5281/zenodo.4739739).After filtering isolated cells and restricting the feature space to genes present in the human LR reference atlas, 159120 cells and 128 genes were retained for analysis. The number of latent communication programs is set to $R=17$.

In [1]:
import anndata as ad
import os
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import CommuNiche as CN

D:\Users\lihs\anaconda3\envs\CommuNiche_test\lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


ModuleNotFoundError: No module named 'CommuNiche'

In [4]:
##### download the dataset and give the datapath to the data_path_ori and data_path_lr
data_path_ori = 'D:/Experiment/Communication_domain/experiment_version3/BC/dataset/BC_adata_original.h5ad'
data_path_lr = 'D:/Experiment/Communication_domain/experiment_version3/BC/dataset/lr_prior_filter.txt'
BC = sc.read_h5ad(data_path_ori)
CN.utils.normalize_then_clip(BC, 95)
lr_prior = pd.read_table(data_path_lr, sep = '\t')
lr_prior['from'] = lr_prior['from'].astype(str)
lr_prior['to'] = lr_prior['to'].astype(str)

gene_all = BC.var_names.values
Ligand_all = lr_prior['from'].unique()
Receptor_all = lr_prior['to'].unique()
intersect_genes = np.intersect1d(np.union1d(Ligand_all, Receptor_all), gene_all)
BC_filter = BC[:, list(intersect_genes)].copy()
l_index = CN.utils.search_index(arr1=intersect_genes, arr2 = lr_prior['from'])
r_index = CN.utils.search_index(arr1=intersect_genes, arr2 = lr_prior['to'])
lr_prior_index = CN.utils.integrate_set(l_index, r_index)
lr_prior_filter = lr_prior.iloc[lr_prior_index]
spatial_coordinate = BC_filter.obsm['spatial']
ad_matrix, _, _, _, _, _ = CN.utils.cal_K_neighboorhood(spatial_coordinate, k=10)
row_sums = ad_matrix.getnnz(axis=1)
nonzero_rows = np.where(row_sums != 0)[0]
BC_filter = BC_filter[nonzero_rows, :]


cell_type_counts = BC_filter.obs['cell_type'].value_counts()
valid_cell_types = cell_type_counts[cell_type_counts > 30].index
BC_filter = BC_filter[BC_filter.obs['cell_type'].isin(valid_cell_types)].copy()

BC = BC[BC_filter.obs_names]
print(BC)
print(BC_filter)

View of AnnData object with n_obs × n_vars = 159120 × 313
    obs: 'cell_type', 'spatial_x', 'spatial_y'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'log1p'
    obsm: 'spatial'
    layers: 'norm', 'norm_clipped'
AnnData object with n_obs × n_vars = 159120 × 40
    obs: 'cell_type', 'spatial_x', 'spatial_y'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'log1p'
    obsm: 'spatial'
    layers: 'norm', 'norm_clipped'


## Construction of communication tensor
In this experiment, we focus the sender-mode for CommuNiche, the user could obtain the different mode communication tensor by change the parameter *mode* in the *CN.utils.cal_cell_L_R* . Besides, the user could select whether smoothing the dataset by parameter *smoothing_ct* and normalize distance 'normalization_dist'.

In [6]:
O_l_single, _, _, cell_names, cell_type_names, lr_pair = CN.utils.cal_cell_L_R(adata = BC_filter, lr_prior = lr_prior_filter, smoothing_ct = True, 
                                                                                   normalization_dist = True,  cell_type_col = 'cell_type',
                                                                                   k = 100)
print(O_l_single.shape)

Calculation of cell-cell neighborhood matrix....
Smoothing the gene expression data....
Calculation of CCI....
Calculation of cell ligands and receptor signal
Calculation of cell type abundace
Joint the tensor and matrix
(159120, 19, 40)


## Tensor decomposition
To identify recurrent communication structure, the tensor is decomposed using constrained non-negative CANDECOMP/PARAFAC (CP) factorization:
$\mathcal{T} \approx \sum_{r=1}^{R} \lambda_r \, U_{:,r} \circ V_{:,r} \circ W_{:,r}$, where $\lambda_r \in \mathbb{R}_{+}$ is the weight of factor $r$, $U \in \mathbb{R}_{+}^{N \times R}$ contains cell loadings, $V \in \mathbb{R}_{+}^{K \times R}$ contains neighboring cell-type loadings, and $W \in \mathbb{R}_{+}^{T \times R}$ contains LR loadings. Here, $\circ$ denotes the vector outer product. Each factor $r$ is interpreted as a latent communication program linking three complementary levels of organization. Specifically, $U_{:,r}$ quantifies program activity across cells, $V_{:,r}$ identifies the neighboring cell populations preferentially associated with the program, and $W_{:,r}$ defines the LR channels contributing to that program.

To improve interpretability and identifiability, three constraints are imposed: (i) non-negativity of all loading matrices, yielding additive parts-based representations; (ii) unit-norm normalization of $V_{:,r}$ and $W_{:,r}$ to resolve scale ambiguity and facilitate comparison across factors; and (iii) a soft orthogonality constraint on $U$ ($U^{\top}U \approx I$), encouraging distinct and minimally overlapping communication programs across cells.

In [9]:
## select the rank by the following
#rank_list, fit_loss, loss_ranks = elbow_selection_large(X = O_l_single, rank_min = 10, rank_max = 20, gamma = 0.1, A_pgd_iters = 12, A_inner = 3, 
#                                                        hals_inner = 4, epochs = 20)
#from tensor_cp_pytorch import find_knee_kneed
#knee_selected = find_knee_kneed(rank_list = rank_list, fit_ranks = fit_loss, plot=True, save_fig = rank_select_file)
#print(f'Knee selected: {knee_selected}')
# rank = 16
A, B, C, lambda_, _, _, _, _ = CN.cp.train_cp_decomposition_large_cells(
        X=O_l_single,
        rank=16,
        sample_ratio=0.1,
        subset_iters= 200,
        hals_inner = 4,
        # training
        epochs = 20,
        block_rows = 50_000,
        A_inner = 3,
        A_pgd_iters= 12)
U_l = A.detach().cpu().numpy()
print(U_l.shape)
V_l = B.detach().cpu().numpy()
print(V_l.shape)
W_l = C.detach().cpu().numpy()
print(W_l.shape)

[Stage 0] subset init | sample_ratio=10.00% | rank=16
reconstruction error=0.6232287106449454
iteration 1, reconstruction error: 0.47129091683212365, decrease = 0.1519377938128218
iteration 2, reconstruction error: 0.4308830580787428, decrease = 0.040407858753380876
iteration 3, reconstruction error: 0.41601626030637534, decrease = 0.014866797772367435
iteration 4, reconstruction error: 0.40807711624734894, decrease = 0.007939144059026404
iteration 5, reconstruction error: 0.4035574758715617, decrease = 0.004519640375787226
iteration 6, reconstruction error: 0.40072408166201157, decrease = 0.002833394209550144
iteration 7, reconstruction error: 0.3987602993416196, decrease = 0.00196378232039196
iteration 8, reconstruction error: 0.3974347284874055, decrease = 0.0013255708542140865
iteration 9, reconstruction error: 0.3964865828648265, decrease = 0.0009481456225789997
iteration 10, reconstruction error: 0.39575536606254064, decrease = 0.0007312168022858834
iteration 11, reconstruction e

## Niche identification
Communication-defined niches are identified by clustering cells in the embedding space $U$, where each cell is represented by its loadings across $R$ latent communication programs. To obtain robust and hierarchically structured partitions, we use a consensus-based clustering strategy (Supplementary Figure~S1b). Louvain clustering is performed on $U$ across multiple resolutions and random initializations. The resulting partitions are aggregated into a co-clustering matrix $\mathcal{C}$, where $\mathcal{C}_{ij}$ records the frequency with which cells $i$ and $j$ are assigned to the same cluster across runs. Hierarchical clustering of $\mathcal{C}$ is then used to derive multilevel niche assignments. The resulting hierarchy defines coarse niches corresponding to broad communication domains and finer sub-niches corresponding to more specific communication states.

For datasets exceeding 100,000 cells, direct construction and clustering of the full co-clustering matrix become computationally expensive. To improve scalability, cells are first grouped into fine-grained micro-clusters using mini-batch K-means. For example, approximately 8,000 micro-clusters are used for datasets containing around 100,000 cells. Each micro-cluster is represented by the mean embedding of its member cells, and consensus clustering is then performed at the meta-cell level. Final niche labels are propagated back to individual cells according to micro-cluster membership. This procedure substantially reduces computational cost while preserving communication-defined structure, enabling application to atlas-scale spatial datasets.

In [12]:
### The K_list are setting for the exploration of hierarchical structure.
K_list = list([5, 9, 13, 17])    
   
Z_meta, labels_by_K_meta, similarity_mat_meta, label1_cells, labels_by_K = CN.utils.louvain_clustering_O_CC_large_data(BC_filter, U_l, K_list,
                                                                                                              mbk_n_clusters = 8000, 
                                                                                                              min_meta_size = 1, interval = 0.01, 
                                                                                                              times_random_running = 5)
BC_filter.obs['CD_clustering_5'] = labels_by_K[5]
BC_filter.obs['CD_clustering_5'] = BC_filter.obs['CD_clustering_5'].astype('category')
    
BC_filter.obs['CD_clustering_9'] = labels_by_K[9]
BC_filter.obs['CD_clustering_9'] = BC_filter.obs['CD_clustering_9'].astype('category')
    
BC_filter.obs['CD_clustering_13'] = labels_by_K[13]
BC_filter.obs['CD_clustering_13'] = BC_filter.obs['CD_clustering_13'].astype('category')

BC_filter.obs['CD_clustering_17'] = labels_by_K[17]
BC_filter.obs['CD_clustering_17'] = BC_filter.obs['CD_clustering_17'].astype('category')



import re
def convert_cluster_to_niche(x):
    if pd.notnull(x):
        match = re.search(r"\d+", str(x))
        if match:
            return f'Niche {int(match.group()) - 1}'
    return x

BC_filter.obs['CD_clustering_5'] = BC_filter.obs['CD_clustering_5'].apply(convert_cluster_to_niche)
BC_filter.obs['CD_clustering_9'] = BC_filter.obs['CD_clustering_9'].apply(convert_cluster_to_niche)
BC_filter.obs['CD_clustering_13'] = BC_filter.obs['CD_clustering_13'].apply(convert_cluster_to_niche)
BC_filter.obs['CD_clustering_17'] = BC_filter.obs['CD_clustering_17'].apply(convert_cluster_to_niche)
BC_filter.obs['CD_clustering_5_vis'] = BC_filter.obs['CD_clustering_5'].str.replace('Niche ', '', regex=False).astype('int64').astype('category')  
BC_filter.obs['CD_clustering_9_vis'] = BC_filter.obs['CD_clustering_9'].str.replace('Niche ', '', regex=False).astype('int64').astype('category')  
BC_filter.obs['CD_clustering_13_vis'] = BC_filter.obs['CD_clustering_13'].str.replace('Niche ', '', regex=False).astype('int64').astype('category')  
BC_filter.obs['CD_clustering_17_vis'] = BC_filter.obs['CD_clustering_17'].str.replace('Niche ', '', regex=False).astype('int64').astype('category')  


BC_filter.obs['CD_clustering_5'] = BC_filter.obs['CD_clustering_5'].astype('category')
BC_filter.obs['CD_clustering_9'] = BC_filter.obs['CD_clustering_9'].astype('category')
BC_filter.obs['CD_clustering_13'] = BC_filter.obs['CD_clustering_13'].astype('category')
BC_filter.obs['CD_clustering_17'] = BC_filter.obs['CD_clustering_17'].astype('category')
BC_filter.obs['CD_clustering_5_vis'] = BC_filter.obs['CD_clustering_5_vis'].astype('category')
BC_filter.obs['CD_clustering_9_vis'] = BC_filter.obs['CD_clustering_9_vis'].astype('category')
BC_filter.obs['CD_clustering_13_vis'] = BC_filter.obs['CD_clustering_13_vis'].astype('category')
BC_filter.obs['CD_clustering_17_vis'] = BC_filter.obs['CD_clustering_17_vis'].astype('category')

Staring Kmeans running----
Kmeans running successfully....
[Init-MBK] 固定 8000 个 meta-cells，得到 8000 个（去重后）。


ModuleNotFoundError: No module named 'igraph'